# Generating and Writing a Trial Wavefunction
---


<font size="5"><b>Goal:</b>
Become acquainted with how to write a Trial wavefunction to the SAFIRE HDF5 format.
</font>

&nbsp;

## What you will learn
---

<font size="4">

1. How to write a single Slater determinant trial wavefunction given the Slater Matrix
2. How to write a single Slater determinant given orbtial occupancies
3. How to write a non-orthogonal multi-Slater determinant trial wavefunction given the expansion coefficients and Slater matrices
</font>

&nbsp;



The trial wavefunction used in lattice models is still constructed as a combination of Slater determinants. In this section, we begin by demonstrating how to manually generate wavefunction inputs and examine the structure of the resulting HDF5 files. This helps build an understanding of what is actually done behind the scenes when preparing a trial wavefunction.

While this manual approach provides clarity and flexibility—especially useful for customizing your own trial wavefunction or performing self-consistent calculations—it is **not** the only method and **not** the recommended way for most practical purposes. In fact, there are several much simpler and more elegant ways to generate trial wavefunctions for certain lattice systems. For example, one can automatically construct free-electron trial states or mean-field trial wavefunctions using the `autoHF` utility, **which will be introduced later.**

To start, we will take a closer look at the `write_wfn` function, which writes the wavefunction data to disk. We will introduce the basic types of wavefunctions—RHF, UHF, and GHF—and explain the corresponding input structures required for each. Understanding this foundation will make it easier to interpret or modify wavefunctions generated by higher-level tools later on.

1. Generate a wavefunction of the form $\Psi = \Psi^\uparrow \otimes \Psi^\downarrow$ with $\Psi^\downarrow_{ij} = \Psi^\uparrow_{ij}$. This means the two spin sectors are identical. Such a wavefunction is commonly referred to as an "RHF" (Restricted Hartree-Fock) wavefunction, and the corresponding walker is called an "RHF walker."

In [1]:
import numpy as np

from afqmctools.wavefunction.model import write_wfn

nx = 2
ny = 2
nelec = (2,2)
assert (nelec[0]==nelec[1])
norb = nx * ny
wfn_input_filename = "wfn_rhf_test.h5"
sd_rhf = np.arange(0,norb * sum(nelec)//2, 1).reshape([1, norb, sum(nelec)//2])
ci = np.ones([1],dtype = np.float64)
write_wfn(filename=wfn_input_filename, wfn=[ci,sd_rhf],
          walker_type='rhf',
          nelec=nelec,
          norb=norb)

If you use `!h5dump wfn_rhf_test.h5` to inspect the output file, you'll see the following structure. The `Psi0_alpha` dataset contains the RHF Slater determinant matrix, which is used for both spin-up and spin-down electrons.

In summary, to generate an RHF wavefunction, you need to provide an array `A[0, i, j]` = $\Psi_{ij}^↑$ with shape `(1, norb, nelec // 2)` as input. Setting walker_type='rhf' ensures the wavefunction is constructed in a spin-identical manner, meaning the same orbitals are used for both spin components.

In [2]:
# HDF5 "wfn_rhf_test.h5" {
# GROUP "/" {
#    GROUP "Wavefunction" {
#       GROUP "NOMSD" {
#          DATASET "Psi0_alpha" {
#             DATATYPE  H5T_IEEE_F64LE
#             DATASPACE  SIMPLE { ( 4, 2, 2 ) / ( 4, 2, 2 ) }
#             DATA {
#             (0,0,0): 0, 0,
#             (0,1,0): 1, 0,
#             (1,0,0): 2, 0,
#             (1,1,0): 3, 0,
#             (2,0,0): 4, 0,
#             (2,1,0): 5, 0,
#             (3,0,0): 6, 0,
#             (3,1,0): 7, 0
#             }
#          }
# ......

2. Generate a wavefunction of the form $\Psi = \Psi^\uparrow \otimes \Psi^\downarrow$, where $\Psi^\downarrow_{ij}$ can differ from $\Psi^\uparrow_{ij}$. This means the two spin sectors are separate and may have distinct wavefunctions. Such a wavefunction is commonly referred to as a "UHF" (Unrestricted Hartree-Fock) wavefunction, and the corresponding walker is called a "UHF walker."

In [3]:
import numpy as np

from afqmctools.wavefunction.model import write_wfn

nx = 2
ny = 2
nelec = (2, 3)
norb = nx * ny
wfn_input_filename = "wfn_uhf_test.h5"
sd_uhf = np.arange(0, norb * sum(nelec), 1).reshape([1, norb, sum(nelec)])
ci = np.ones([1], dtype = np.float64)
write_wfn(filename=wfn_input_filename, wfn=[ci, sd_uhf],
          walker_type='uhf',
          nelec=nelec,
          norb=norb)

/mnt/home/beskridge/software/SAFIRE/utils/afqmctools/wavefunction/common.py:176: UserWarning: Using UHF Slater determinant for initial Walkers with a Collinear Trial Wavefunction. This can lead to very slow equilibration in AFQMC calculations. using ROHF Slater determinants for intiail Walkers is recommended.
  warn(


If you use `!h5dump wfn_uhf_test.h5` to inspect the output file, you'll see the following structure. The `Psi0_alpha` dataset contains the Slater determinant matrix for spin-up electrons, while `Psi0_beta` contains the matrix for spin-down electrons.

In summary, to generate a UHF wavefunction, you need to provide an array `A` of shape `(1, norb, nelec[0] + nelec[1])`, where `A[0, i, j]` = $\Psi_{ij}^↑$ for j < nelec[0], and `A[0, i, nelec[0] + j]` = $\Psi_{ij}^↓$ for j < nelec[1]. Setting walker_type='uhf' ensures the wavefunction is constructed with separate spin-up and spin-down components.

In [4]:
# HDF5 "wfn_uhf_test.h5" {
# GROUP "/" {
#    GROUP "Wavefunction" {
#       GROUP "NOMSD" {
#          DATASET "Psi0_alpha" {
#             DATATYPE  H5T_IEEE_F64LE
#             DATASPACE  SIMPLE { ( 4, 2, 2 ) / ( 4, 2, 2 ) }
#             DATA {
#             (0,0,0): 0, 0,
#             (0,1,0): 1, 0,
#             (1,0,0): 5, 0,
#             (1,1,0): 6, 0,
#             (2,0,0): 10, 0,
#             (2,1,0): 11, 0,
#             (3,0,0): 15, 0,
#             (3,1,0): 16, 0
#             }
#          }
#          DATASET "Psi0_beta" {
#             DATATYPE  H5T_IEEE_F64LE
#             DATASPACE  SIMPLE { ( 4, 3, 2 ) / ( 4, 3, 2 ) }
#             DATA {
#             (0,0,0): 2, 0,
#             (0,1,0): 3, 0,
#             (0,2,0): 4, 0,
#             (1,0,0): 7, 0,
#             (1,1,0): 8, 0,
#             (1,2,0): 9, 0,
#             (2,0,0): 12, 0,
#             (2,1,0): 13, 0,
#             (2,2,0): 14, 0,
#             (3,0,0): 17, 0,
#             (3,1,0): 18, 0,
#             (3,2,0): 19, 0
#             }
#          }
# ........

Generate a wavefunction of the general form $\Psi = \Psi^{\uparrow,\downarrow}$, where the spin sectors are mixed. This form typically arises in systems with spin-orbit coupling or magnetic order in the $xy$ plane. Such wavefunctions are commonly referred to as "GHF" (Generalized Hartree-Fock) wavefunctions, and the associated walkers are called "GHF walkers."

In [5]:
import numpy as np

from afqmctools.wavefunction.model import write_wfn

nx = 2
ny = 2
nelec = (4, 0)
# (3, 1) or (2, 2) or (1, 3) or (0,4) are equivalent are all equivalent,
# as long as the total number of electrons is the same
# This is because the GHF wavefunction does not distinguish between spin-up and spin-down components.
norb = nx * ny
wfn_input_filename = "wfn_ghf_test.h5"
sd_uhf = np.arange(0, 2 * norb * sum(nelec), 1).reshape([1, 2 * norb, sum(nelec)])
ci = np.ones([1], dtype = np.float64)
write_wfn(filename=wfn_input_filename, wfn=[ci, sd_uhf],
          walker_type='ghf',
          nelec=nelec,
          norb=norb)

If you inspect the output file using !h5dump wfn_ghf_test.h5, you'll see the following structure. Notably, there is only a `Psi0_alpha` dataset, but its shape is `[2 * norb, sum(nelec)]`, representing the full GHF Slater determinant matrix with both spin sectors mixed.
In summary, to generate a GHF wavefunction, you need to provide an array `A[0, i, j]` = $\Psi_{ij}$ with shape `(1, 2 * norb, nelec[0] + nelec[1])` as input. Setting walker_type='ghf' ensures the wavefunction is treated in a mixed spin manner. Note that the `norb` parameter here still refers to the number of spatial orbitals per spin.

In [6]:
# HDF5 "wfn_ghf_test.h5" {
# GROUP "/" {
#    GROUP "Wavefunction" {
#       GROUP "NOMSD" {
#          DATASET "Psi0_alpha" {
#             DATATYPE  H5T_IEEE_F64LE
#             DATASPACE  SIMPLE { ( 8, 4, 2 ) / ( 8, 4, 2 ) }
#             DATA {
#             (0,0,0): 0, 0,
#             (0,1,0): 1, 0,
#             (0,2,0): 2, 0,
#             (0,3,0): 3, 0,
#             (1,0,0): 4, 0,
#             (1,1,0): 5, 0,
#             (1,2,0): 6, 0,
#             (1,3,0): 7, 0,
#             (2,0,0): 8, 0,
#             (2,1,0): 9, 0,
#             (2,2,0): 10, 0,
#             (2,3,0): 11, 0,
#             (3,0,0): 12, 0,
#             (3,1,0): 13, 0,
#             (3,2,0): 14, 0,
#             (3,3,0): 15, 0,
#             (4,0,0): 16, 0,
#             (4,1,0): 17, 0,
#             (4,2,0): 18, 0,
#             (4,3,0): 19, 0,
#             (5,0,0): 20, 0,
#             (5,1,0): 21, 0,
#             (5,2,0): 22, 0,
#             (5,3,0): 23, 0,
#             (6,0,0): 24, 0,
#             (6,1,0): 25, 0,
#             (6,2,0): 26, 0,
#             (6,3,0): 27, 0,
#             (7,0,0): 28, 0,
#             (7,1,0): 29, 0,
#             (7,2,0): 30, 0,
#             (7,3,0): 31, 0
#             }
#          }
# ........

4. Multi-Slater trial wavefunctions are also supported in lattice models. To enable this, the input variable `ci` is provided, which is a 1D array representing the coefficients of the Slater determinants. Correspondingly, the Slater determinant inputs include an extra dimension to stack multiple Slater matrices. To generate a multi-Slater trial wavefunction, simply supply the `ci` array as the list of coefficients and stack the individual Slater matrices along the first dimension—everything else will be handled automatically.

In [7]:
import numpy as np

from afqmctools.wavefunction.model import write_wfn

nx = 2
ny = 2
nelec = (4, 0) # (3, 1) or (2, 2) or (1, 3) or (0,4) are equivalent as long as sum(nelec) is the same
norb = nx * ny
wfn_input_filename = "wfn_multi_ghf_test.h5"
sd_uhf = np.arange(0, 3 * 2 * norb * sum(nelec), 1).reshape([3, 2 * norb, sum(nelec)])
ci = np.array([1, -1, 4], dtype = np.float64)
write_wfn(filename=wfn_input_filename, wfn=[ci, sd_uhf],
          walker_type='ghf',
          nelec=nelec,
          norb=norb)

with `!h5ls -r -d wfn_multi_ghf_test.h5/Wavefunction/NOMSD ` you will see an output similar to the following, indicating that there are three datasets named `PsiT_0`, `PsiT_1`, and `PsiT_2` in the file.

In [8]:
# /Psi0_alpha              Dataset {8, 4, 2}
#     Data:
#          0, 0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 0, 6, 0, 7, 0, 8, 0, 9, 0, 10, 0, 11,
#          0, 12, 0, 13, 0, 14, 0, 15, 0, 16, 0, 17, 0, 18, 0, 19, 0, 20, 0, 21,
#          0, 22, 0, 23, 0, 24, 0, 25, 0, 26, 0, 27, 0, 28, 0, 29, 0, 30, 0, 31, 0
# /PsiT_0                  Group
# /PsiT_0/data_            Dataset {31, 2}
#     Data:
#          4, -0, 8, -0, 12, -0, 16, -0, 20, -0, 24, -0, 28, -0, 1, -0, 5, -0, 9,
#          -0, 13, -0, 17, -0, 21, -0, 25, -0, 29, -0, 2, -0, 6, -0, 10, -0, 14,
#          -0, 18, -0, 22, -0, 26, -0, 30, -0, 3, -0, 7, -0, 11, -0, 15, -0, 19,
#          -0, 23, -0, 27, -0, 31, -0
# /PsiT_0/dims             Dataset {3}
#     Data:
#          4, 8, 31
# /PsiT_0/jdata_           Dataset {31}
#     Data:
#          1, 2, 3, 4, 5, 6, 7, 0, 1, 2, 3, 4, 5, 6, 7, 0, 1, 2, 3, 4, 5, 6, 7, 0,
#          1, 2, 3, 4, 5, 6, 7
# /PsiT_0/pointers_begin_  Dataset {4}
#     Data:
#          0, 7, 15, 23
# /PsiT_0/pointers_end_    Dataset {4}
#     Data:
#          7, 15, 23, 31
# /PsiT_1                  Group
# /PsiT_1/data_            Dataset {32, 2}
#     Data:
#          32, -0, 36, -0, 40, -0, 44, -0, 48, -0, 52, -0, 56, -0, 60, -0, 33, -0,
#          37, -0, 41, -0, 45, -0, 49, -0, 53, -0, 57, -0, 61, -0, 34, -0, 38, -0,
#          42, -0, 46, -0, 50, -0, 54, -0, 58, -0, 62, -0, 35, -0, 39, -0, 43, -0,
#          47, -0, 51, -0, 55, -0, 59, -0, 63, -0
# /PsiT_1/dims             Dataset {3}
#     Data:
#          4, 8, 32
# /PsiT_1/jdata_           Dataset {32}
#     Data:
#          0, 1, 2, 3, 4, 5, 6, 7, 0, 1, 2, 3, 4, 5, 6, 7, 0, 1, 2, 3, 4, 5, 6, 7,
#          0, 1, 2, 3, 4, 5, 6, 7
# /PsiT_1/pointers_begin_  Dataset {4}
#     Data:
#          0, 8, 16, 24
# /PsiT_1/pointers_end_    Dataset {4}
#     Data:
#          8, 16, 24, 32
# /PsiT_2                  Group
# /PsiT_2/data_            Dataset {32, 2}
#     Data:
#          64, -0, 68, -0, 72, -0, 76, -0, 80, -0, 84, -0, 88, -0, 92, -0, 65, -0,
#          69, -0, 73, -0, 77, -0, 81, -0, 85, -0, 89, -0, 93, -0, 66, -0, 70, -0,
#          74, -0, 78, -0, 82, -0, 86, -0, 90, -0, 94, -0, 67, -0, 71, -0, 75, -0,
#          79, -0, 83, -0, 87, -0, 91, -0, 95, -0
# /PsiT_2/dims             Dataset {3}
#     Data:
#          4, 8, 32
# /PsiT_2/jdata_           Dataset {32}
#     Data:
#          0, 1, 2, 3, 4, 5, 6, 7, 0, 1, 2, 3, 4, 5, 6, 7, 0, 1, 2, 3, 4, 5, 6, 7,
#          0, 1, 2, 3, 4, 5, 6, 7
# /PsiT_2/pointers_begin_  Dataset {4}
#     Data:
#          0, 8, 16, 24
# /PsiT_2/pointers_end_    Dataset {4}
#     Data:
#          8, 16, 24, 32
# /ci_coeffs               Dataset {3, 2}
#     Data:
#          1, 0, -1, 0, 4, 0
# /dims                    Dataset {5}
#     Data:
#          4, 4, 0, 3, 3

5. (Optional) Change the wavefunction used to initialize the walker.
In the `write_wfn` function, there is an `init` parameter that accepts the initial wavefunction in a format similar to the corresponding `wfn[1]` (where `wfn = [ci, sd_mats]`). The init input should include an additional leading dimension, regardless of whether it is a multi-Slater trial or not. For example:
`[1, norb, nelec//2]` for RHF, `[1, norb, sum(nelec)]` for UHF, `[1, 2 * norb, sum(nelec)]` for GHF. **By default**, the initial walker is set to be identical to the trial wavefunction—specifically, the first trial wavefunction if using a multi-Slater trial.

## Writing a trial wavefunction

---

We can generate a free-electron trial wavefunction using the `free_electron` function within `afqmctools`. First, we set up the lattice using the `get_lattice` function (in this case a square 4 x 4 lattice with periodic boundary conditions in both directions) and then define other relevant parameters (`nelec`, `Uhubb`, `hopping`). We then build the Hamiltonian that we will simulate with AFQMC. Finally, the trial wavefunction is computed from the one-body part of the Hamiltonian and written in HDF5 format to the file "wfn.h5".

In [9]:
from afqmctools.systems.lattice import get_lattice
from afqmctools.hamiltonian.model.builder import HamiltonianBuilder
from afqmctools.hamiltonian.model.ham_class import HamiltonianComponent, SpinSymm
from afqmctools.utils.io import write_model_hamiltonian
from afqmctools.wavefunction.free_electron import free_electron
from afqmctools.wavefunction.model import write_wfn

# define lattice
lattice = get_lattice(
    params=dict(
        L1 = 4,
        L2 = 4,
        boundary1 = "PBC",
        boundary2 = "PBC"
    )
)

nelec = (5,5)
Uhubb = 4.0

hopping = [1.0,0.0]

builder = HamiltonianBuilder(
    lattice=lattice,
    spin_symm=SpinSymm.COLLINEAR
)
# add standard Hubbard terms
builder.nth_neighbor_hopping(hopping)
builder.onsite_hubbard(Uhubb)

nbasis = lattice.N_sites

builder.finalize()

hamiltonian = builder.hamiltonian

write_model_hamiltonian(
    hamiltonian=hamiltonian,
    fname="hamil.h5",
    nelec=nelec
)

# compute a free-electron trial wfn
wfn,spin_symm = free_electron(
    source=hamiltonian,
    lattice=lattice,
    nelec=nelec
)

# and write the wavefunction for use in SAFIRE
write_wfn(
    filename="wfn.h5",
    wfn=wfn,
    walker_type=spin_symm,
    nelec=nelec,
    norb=lattice.N_sites
)

/mnt/home/beskridge/software/SAFIRE/utils/afqmctools/wavefunction/free_electron.py:126: UserWarning: Twist angle of the Hamiltonian does not match the twist angle provided to the free electron wavefunction builder. Using the twist angle from the Hamiltonian.
  warn(


Calling nth_neighbor_hopping with 1.0 for nth_neighbor=1
computing and storing 1th-nearest neighbors
Computing distance matrix of lattice
Reading lattice site positions from Lattice
computing and storing 1th-nearest image neighbors
Using same twists for up and down spins
Calling nth_neighbor_hopping with 0.0 for nth_neighbor=2
Building Hubbard U term with U=4.0
Building onsite hubbard with positive U values: 4.0
Combining terms of the same type
Combining Hubbard U, U1, and U2 terms where possible
Max spin symmetry is  2
Generating free-electron trial wavefunction with twist = [4.10803005e-06 4.58334805e-04]

Processing spin-up channel (5 electrons)
using dense representation of 1-body Hamiltonian to find eigenvalues and orbitals

Eigenvalues of the non-interacting Hamiltonian:
[-4.00000000e+00 -2.00000000e+00 -2.00000000e+00 -2.00000000e+00
 -2.00000000e+00 -3.28166882e-15  2.03437729e-16  2.83280437e-16
  2.66453526e-15  2.66453526e-15  6.92556512e-15  2.00000000e+00
  2.00000000e+00 

/mnt/home/beskridge/software/AutoHF_dev/autohf/solver.py:911: ComplexWarning: Casting complex values to real discards the imaginary part
  state0_ref[i, 0:x, 0:y] = p[:, :]


energyCall: Etotal=-17.75 with EK=-24.0 EU=6.250000000000001 EU1=0 EU2=0 EJ=0 Eheisenber=0
Reference HF Energy = -17.75
<Sz> 0.0 <S^2> 0.0


/mnt/home/beskridge/software/SAFIRE/utils/afqmctools/wavefunction/common.py:176: UserWarning: Using UHF Slater determinant for initial Walkers with a Collinear Trial Wavefunction. This can lead to very slow equilibration in AFQMC calculations. using ROHF Slater determinants for intiail Walkers is recommended.
  warn(


# Summary
---

<font size="4">

In this tutorial, you became acquainted with how to write a Trial wavefunction to the SAFIRE HDF5 format.

</font>

&nbsp;

## What you learned
---

<font size="4">

1. How to write a single Slater determinant trial wavefunction given the Slater Matrix
2. How to write a single Slater determinant given orbtial occupancies
3. How to write a non-orthogonal multi-Slater determinant trial wavefunction given the expansion coefficients and Slater matrices

</font>

&nbsp;
